# HIG publication tables from the rectangle benchmark
This notebook reproduces the 1300-image controlled synthetic experiment and exports the filled CSV tables plus a publication-ready LaTeX Section 6.


In [ ]:
!pip -q install pandas pillow gradio


In [ ]:
import os, math, tempfile, zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import gradio as gr

# ============================================================
# Global configuration
# ============================================================
SIZE = 10
OUT = -1
COLOR_NAMES = ["red", "blue", "green", "yellow", "purple"]
COLOR_RGB = [
    (231, 76, 60),
    (52, 152, 219),
    (46, 204, 113),
    (241, 196, 15),
    (155, 89, 182),
]
DIR4 = [(-1,0),(1,0),(0,-1),(0,1)]
DIR8 = [(dr,dc) for dr in (-1,0,1) for dc in (-1,0,1) if (dr,dc)!=(0,0)]

# Cubical 10x10 model sizes used for the compression ratios.
K0 = (SIZE + 1) * (SIZE + 1)   # 121
K1 = SIZE * (SIZE + 1) + (SIZE + 1) * SIZE  # 220
K2 = SIZE * SIZE               # 100
CUBICAL_CELLS = K0 + K1 + K2   # 441
# Support size of M01 plus compactified M12: 2 incidences per grid edge in each level.
CUBICAL_INCIDENCE_SUPPORT = 4 * K1  # 880

# ============================================================
# Structured rectangle generator
# ============================================================

def even_parts(total, parts):
    q, rem = divmod(total, parts)
    return [q + (i < rem) for i in range(parts)]


def layout_tiles(k, size=SIZE):
    """Candidate near-grid layouts for k root rectangles."""
    layouts = []
    for rows in range(1, k + 1):
        cols = math.ceil(k / rows)
        if rows > size or cols > size:
            continue
        hs = even_parts(size, rows)
        ws = even_parts(size, cols)
        tiles = []
        idx = 0
        y = 0
        for i, h in enumerate(hs):
            x = 0
            for j, w in enumerate(ws):
                if idx < k:
                    tiles.append((y, x, h, w, i, j))
                    idx += 1
                x += w
            y += h
        # A strict child needs a one-pixel margin on every side.
        capacities = [max(0, (min(h, w) - 1) // 2) for y,x,h,w,i,j in tiles]
        layouts.append((sum(capacities), rows, cols, tiles, capacities))
    return layouts


def max_nested_capacity(root_count, size=SIZE):
    if root_count < 1:
        return -1
    layouts = layout_tiles(root_count, size)
    return max((x[0] for x in layouts), default=-1)


def feasible_pair(r, n, size=SIZE):
    if r < 1 or n < 0 or n >= r:
        return False
    return n <= max_nested_capacity(r - n, size)


def choose_layout_and_depths(r, n, rng, size=SIZE):
    roots = r - n
    feasible = []
    for item in layout_tiles(roots, size):
        capsum, rows, cols, tiles, caps = item
        if capsum >= n:
            feasible.append(item)
    if not feasible:
        raise ValueError(
            f"(r={r}, n={n}) is not feasible on a {size}x{size} grid with strict one-pixel nesting margins."
        )
    # Prefer layouts with good nesting capacity but keep some variation.
    feasible.sort(key=lambda z: z[0], reverse=True)
    best = feasible[0][0]
    candidates = [z for z in feasible if z[0] >= max(n, best - 1)]
    capsum, rows, cols, tiles, caps = candidates[int(rng.integers(len(candidates)))]

    depths = [0] * roots
    remaining = n
    while remaining:
        available = [i for i, c in enumerate(caps) if depths[i] < c]
        if not available:
            raise RuntimeError("Internal error allocating nesting depths.")
        i = int(rng.choice(available))
        depths[i] += 1
        remaining -= 1
    return tiles, depths, (rows, cols)


def random_rect_within(tile, min_side, rng):
    y, x, h, w, _, _ = tile
    if min(h, w) < min_side:
        raise ValueError("Tile too small for requested nesting depth.")
    rh = int(rng.integers(min_side, h + 1))
    rw = int(rng.integers(min_side, w + 1))
    y0 = int(rng.integers(y, y + h - rh + 1))
    x0 = int(rng.integers(x, x + w - rw + 1))
    return (y0, x0, rh, rw)


def nested_child_rect(parent, remaining_depth_after_child, rng):
    y, x, h, w = parent
    min_side = 2 * remaining_depth_after_child + 1
    max_h = h - 2
    max_w = w - 2
    if max_h < min_side or max_w < min_side:
        raise ValueError("Parent rectangle is too small for the requested nested chain.")
    rh = int(rng.integers(min_side, max_h + 1))
    rw = int(rng.integers(min_side, max_w + 1))
    y0 = int(rng.integers(y + 1, y + h - rh)) if (y + h - rh) > (y + 1) else y + 1
    x0 = int(rng.integers(x + 1, x + w - rw)) if (x + w - rw) > (x + 1) else x + 1
    return (y0, x0, rh, rw)


def generate_rectangular_segmentation(r, n, seed, size=SIZE):
    """Generate one 10x10 five-color image with exactly r drawn rectangles, n nested."""
    if not feasible_pair(r, n, size):
        raise ValueError(
            f"Invalid/infeasible parameters r={r}, n={n}. "
            f"Require 0 <= n < r and sufficient space for strict nesting."
        )

    rng = np.random.default_rng(int(seed))
    A = np.zeros((size, size), dtype=np.int8)  # background = red / label 0
    tiles, depths, layout = choose_layout_and_depths(r, n, rng, size)
    primitives = []

    # Root colors: a randomized 4-color pattern that prevents neighboring tile roots
    # from having the same color when they touch.
    root_palette = list(rng.permutation(np.arange(1, len(COLOR_RGB))))

    for root_id, (tile, depth) in enumerate(zip(tiles, depths)):
        min_side = 2 * depth + 1
        root = random_rect_within(tile, min_side, rng)
        tile_row, tile_col = tile[4], tile[5]
        color = int(root_palette[(tile_row + tile_col) % len(root_palette)])

        y, x, h, w = root
        A[y:y+h, x:x+w] = color
        primitives.append({
            "primitive": len(primitives), "root": root_id, "depth": 0,
            "parent": None, "y": y, "x": x, "h": h, "w": w, "color": color
        })
        parent_idx = len(primitives) - 1
        parent = root
        parent_color = color

        for level in range(1, depth + 1):
            remaining = depth - level
            child = nested_child_rect(parent, remaining, rng)
            choices = [c for c in range(len(COLOR_RGB)) if c != parent_color]
            child_color = int(rng.choice(choices))
            yy, xx, hh, ww = child
            A[yy:yy+hh, xx:xx+ww] = child_color
            primitives.append({
                "primitive": len(primitives), "root": root_id, "depth": level,
                "parent": parent_idx, "y": yy, "x": xx, "h": hh, "w": ww,
                "color": child_color
            })
            parent_idx = len(primitives) - 1
            parent = child
            parent_color = child_color

    assert len(primitives) == r
    assert sum(p["parent"] is not None for p in primitives) == n
    return A, primitives, depths, layout

# ============================================================
# Intrinsic HIG implementation
# ============================================================

def label_4_regions(A):
    h, w = A.shape
    region_id = -np.ones((h, w), dtype=int)
    regions = []
    rid = 0
    for r in range(h):
        for c in range(w):
            if region_id[r, c] != -1:
                continue
            color = int(A[r, c])
            stack = [(r, c)]
            region_id[r, c] = rid
            pixels = set()
            while stack:
                x, y = stack.pop()
                pixels.add((x, y))
                for dx, dy in DIR4:
                    nx, ny = x + dx, y + dy
                    if (0 <= nx < h and 0 <= ny < w and
                        region_id[nx, ny] == -1 and int(A[nx, ny]) == color):
                        region_id[nx, ny] = rid
                        stack.append((nx, ny))
            regions.append({"id": rid, "color": color, "pixels": pixels})
            rid += 1
    return region_id, regions


def edge_key(a, b):
    return tuple(sorted((tuple(a), tuple(b))))


def build_frontier(A, region_id):
    h, w = A.shape
    edge_cells = defaultdict(list)
    for r in range(h):
        for c in range(w):
            pixel_edges = [
                ((r,c),(r,c+1)), ((r+1,c),(r+1,c+1)),
                ((r,c),(r+1,c)), ((r,c+1),(r+1,c+1))
            ]
            for a, b in pixel_edges:
                edge_cells[edge_key(a,b)].append((r,c))

    frontier = []
    vertex_to_edges = defaultdict(list)
    for key, cells in edge_cells.items():
        if len(cells) == 1:
            pair = (OUT, int(region_id[cells[0]]))
        else:
            r1 = int(region_id[cells[0]])
            r2 = int(region_id[cells[1]])
            if r1 == r2:
                continue
            pair = (r1, r2)
        pair = tuple(sorted(pair))
        idx = len(frontier)
        frontier.append({"vertices": key, "cells": list(cells), "pair": pair})
        for v in key:
            vertex_to_edges[v].append(idx)
    return frontier, vertex_to_edges


def frontier_components(frontier, vertex_to_edges):
    comp_of_edge = [-1] * len(frontier)
    comps = []
    for start in range(len(frontier)):
        if comp_of_edge[start] != -1:
            continue
        cid = len(comps)
        comp_of_edge[start] = cid
        stack = [start]
        comp = []
        while stack:
            e = stack.pop()
            comp.append(e)
            for v in frontier[e]["vertices"]:
                for f in vertex_to_edges[v]:
                    if comp_of_edge[f] == -1:
                        comp_of_edge[f] = cid
                        stack.append(f)
        comps.append(comp)
    return comps, comp_of_edge


def continuation_classes(frontier, vertex_to_edges):
    junctions = {v for v, edges in vertex_to_edges.items() if len(edges) >= 3}
    class_of_edge = [-1] * len(frontier)
    classes = []
    for start in range(len(frontier)):
        if class_of_edge[start] != -1:
            continue
        cid = len(classes)
        class_of_edge[start] = cid
        pair = frontier[start]["pair"]
        stack = [start]
        support = []
        while stack:
            e = stack.pop()
            support.append(e)
            for v in frontier[e]["vertices"]:
                if v in junctions:
                    continue
                for f in vertex_to_edges[v]:
                    if class_of_edge[f] == -1 and frontier[f]["pair"] == pair:
                        class_of_edge[f] = cid
                        stack.append(f)

        endpoints = []
        for e in support:
            for v in frontier[e]["vertices"]:
                if v in junctions:
                    endpoints.append(v)

        if len(endpoints) == 0:
            gtype = "enc0"
        elif len(endpoints) == 2:
            gtype = "enc1" if endpoints[0] == endpoints[1] else "seg"
        else:
            raise RuntimeError(f"Invalid continuation class: {len(endpoints)} terminal occurrences.")
        classes.append({"id": f"G{cid}", "type": gtype, "edges": support, "pair": pair})
    return classes, junctions


def holes_of_region(rid, region_id):
    h, w = region_id.shape
    seen = np.zeros((h,w), dtype=bool)
    holes = []
    for r in range(h):
        for c in range(w):
            if region_id[r,c] == rid or seen[r,c]:
                continue
            stack = [(r,c)]
            seen[r,c] = True
            comp = set()
            touches_frame = False
            while stack:
                x,y = stack.pop()
                comp.add((x,y))
                if x in (0,h-1) or y in (0,w-1):
                    touches_frame = True
                for dx,dy in DIR8:
                    nx,ny = x+dx,y+dy
                    if (0 <= nx < h and 0 <= ny < w and not seen[nx,ny]
                        and region_id[nx,ny] != rid):
                        seen[nx,ny] = True
                        stack.append((nx,ny))
            if not touches_frame:
                holes.append(comp)
    return holes


def identify_cuts(regions, region_id, frontier, f_components, comp_of_edge):
    frame_free = [
        cid for cid, comp in enumerate(f_components)
        if all(OUT not in frontier[e]["pair"] for e in comp)
    ]
    cut_map = {}
    for region in regions:
        rid = region["id"]
        for hole in holes_of_region(rid, region_id):
            interface = []
            for eidx, e in enumerate(frontier):
                cells = e["cells"]
                if len(cells) != 2:
                    continue
                p, q = cells
                if ((p in hole and int(region_id[q]) == rid) or
                    (q in hole and int(region_id[p]) == rid)):
                    interface.append(eidx)
            if not interface:
                raise RuntimeError("Regional hole without frontier interface.")
            cids = {comp_of_edge[e] for e in interface}
            if len(cids) != 1:
                raise RuntimeError("Regional hole interface spans more than one frontier component.")
            cid = next(iter(cids))
            if cid in cut_map and cut_map[cid][0] != rid:
                raise RuntimeError("A frame-free frontier component received two carrier regions.")
            cut_map[cid] = (rid, hole)

    if set(frame_free) != set(cut_map):
        raise RuntimeError("Cut-hole bijection check failed in the implementation.")

    return [
        {"id": f"C{cid}", "type": "cut", "component": cid,
         "edges": f_components[cid], "carrier": cut_map[cid][0], "hole": cut_map[cid][1]}
        for cid in frame_free
    ]


def rag_edges(region_id):
    h,w = region_id.shape
    edges = set()
    for r in range(h):
        for c in range(w):
            a = int(region_id[r,c])
            if r+1 < h:
                b = int(region_id[r+1,c])
                if a != b: edges.add(tuple(sorted((a,b))))
            if c+1 < w:
                b = int(region_id[r,c+1])
                if a != b: edges.add(tuple(sorted((a,b))))
    return edges


def compute_hig(A):
    region_id, regions = label_4_regions(A)
    frontier, vertex_to_edges = build_frontier(A, region_id)
    f_components, comp_of_edge = frontier_components(frontier, vertex_to_edges)
    continuation, junctions = continuation_classes(frontier, vertex_to_edges)
    cuts = identify_cuts(regions, region_id, frontier, f_components, comp_of_edge)

    seg  = [g for g in continuation if g["type"] == "seg"]
    enc1 = [g for g in continuation if g["type"] == "enc1"]
    enc0 = [g for g in continuation if g["type"] == "enc0"]
    V1 = seg + enc1 + enc0 + cuts
    V1_types = ["seg"]*len(seg) + ["enc1"]*len(enc1) + ["enc0"]*len(enc0) + ["cut"]*len(cuts)

    junction_list = sorted(junctions)
    V0 = [("junc",v) for v in junction_list] + [("enc",g["id"]) for g in enc0]
    V0_types = ["junc"]*len(junction_list) + ["enc"]*len(enc0)
    V2 = list(range(len(regions))) + [OUT]
    V2_types = ["img"]*len(regions) + ["OUT"]

    M01 = np.zeros((len(V0),len(V1)), dtype=int)
    M12 = np.zeros((len(V1),len(V2)), dtype=int)
    jrow = {v:i for i,v in enumerate(junction_list)}
    erow = {g["id"]:len(junction_list)+i for i,g in enumerate(enc0)}
    v2col = {rid:i for i,rid in enumerate(V2)}

    for j,g in enumerate(V1):
        branch_count = defaultdict(int)
        for eidx in g["edges"]:
            for v in frontier[eidx]["vertices"]:
                if v in junctions:
                    branch_count[v] += 1
        for v,m in branch_count.items():
            M01[jrow[v],j] = m
        if g["type"] == "enc0":
            M01[erow[g["id"]],j] = 2
        if g["type"] == "cut":
            M12[j,v2col[g["carrier"]]] = 2
        else:
            r1,r2 = g["pair"]
            M12[j,v2col[r1]] = 1
            M12[j,v2col[r2]] = 1

    V0n,V1n,V2n = len(V0),len(V1),len(V2)
    chi_cell = V2n - V1n + V0n
    if chi_cell != 2:
        raise AssertionError(f"Cellular Euler check failed: chi_cell={chi_cell}")

    graph_V = V0n + V1n + V2n
    graph_E = int(M01.sum() + M12.sum())
    chi_graph = graph_V - graph_E

    # incidence multigraph connected components (multiplicity does not alter support connectivity)
    adj = [set() for _ in range(graph_V)]
    off1 = V0n
    off2 = V0n + V1n
    for i,j in zip(*np.nonzero(M01)):
        adj[i].add(off1+j); adj[off1+j].add(i)
    for j,k in zip(*np.nonzero(M12)):
        adj[off1+j].add(off2+k); adj[off2+k].add(off1+j)
    seen = set(); components = 0
    for s in range(graph_V):
        if s in seen: continue
        components += 1
        stack=[s]; seen.add(s)
        while stack:
            u=stack.pop()
            for v in adj[u]:
                if v not in seen:
                    seen.add(v); stack.append(v)
    beta1_graph = graph_E - graph_V + components

    redges = rag_edges(region_id)

    return {
        "region_id": region_id, "regions": regions, "frontier": frontier,
        "seg": seg, "enc1": enc1, "enc0": enc0, "cuts": cuts,
        "junctions": junction_list, "V0": V0, "V1": V1, "V2": V2,
        "V0_types": V0_types, "V1_types": V1_types, "V2_types": V2_types,
        "M01": M01, "M12": M12,
        "chi_cell": chi_cell, "graph_V": graph_V, "graph_E": graph_E,
        "chi_graph": chi_graph, "graph_components": components,
        "beta1_graph": beta1_graph, "rag_edges": redges,
    }

# ============================================================
# Statistics and rendering
# ============================================================

def block_stats(M, row_types, col_types, rt, ct):
    rows=[i for i,t in enumerate(row_types) if t==rt]
    cols=[j for j,t in enumerate(col_types) if t==ct]
    if not rows or not cols:
        return 0,0
    B=M[np.ix_(rows,cols)]
    return int(np.count_nonzero(B)), int(B.sum())


def one_row(A, primitives, r, n, seed, image_id):
    h = compute_hig(A)
    M01,M12=h["M01"],h["M12"]
    nregions=len(h["regions"])
    row = {
        "image_id": image_id, "seed": int(seed), "r_rectangles": int(r), "n_nested": int(n),
        "V2_img": nregions, "V2": len(h["V2"]),
        "V1_seg": len(h["seg"]), "V1_enc1": len(h["enc1"]),
        "V1_enc0": len(h["enc0"]), "V1_cut": len(h["cuts"]), "V1": len(h["V1"]),
        "V0_junc": len(h["junctions"]), "V0_enc": len(h["enc0"]), "V0": len(h["V0"]),
        "chi_cell": h["chi_cell"], "V_graph": h["graph_V"], "E_graph": h["graph_E"],
        "chi_graph": h["chi_graph"], "graph_components": h["graph_components"],
        "beta1_graph": h["beta1_graph"],
        "nnz_M01": int(np.count_nonzero(M01)), "nnz_M12": int(np.count_nonzero(M12)),
        "m01": int(M01.sum()), "m12": int(M12.sum()),
        "d01": float(np.count_nonzero(M01)/(M01.size if M01.size else 1)),
        "d12": float(np.count_nonzero(M12)/(M12.size if M12.size else 1)),
        "RAG_V": nregions, "RAG_E": len(h["rag_edges"]),
        "rho_HIG": float((len(h["V0"])+len(h["V1"])+len(h["V2"]))/CUBICAL_CELLS),
        "rho_inc": float((np.count_nonzero(M01)+np.count_nonzero(M12))/CUBICAL_INCIDENCE_SUPPORT),
    }
    return row, h


def interaction_rows(h, image_id, r, n):
    M01,M12=h["M01"],h["M12"]
    t0,t1,t2=h["V0_types"],h["V1_types"],h["V2_types"]
    out=[]
    pairs01=[
        ("V0_junc--V1_seg","junc","seg"),
        ("V0_junc--V1_enc1","junc","enc1"),
        ("V0_junc--V1_enc0","junc","enc0"),
        ("V0_junc--V1_cut","junc","cut"),
        ("V0_enc--V1_enc0","enc","enc0"),
    ]
    for name,a,b in pairs01:
        supp,mult=block_stats(M01,t0,t1,a,b)
        out.append({"image_id":image_id,"r_rectangles":r,"n_nested":n,
                    "interaction":name,"support":supp,"multiplicity":mult})
    pairs12=[
        ("V1_seg--V2_img","seg","img"),("V1_seg--OUT","seg","OUT"),
        ("V1_enc1--V2_img","enc1","img"),("V1_enc1--OUT","enc1","OUT"),
        ("V1_enc0--V2_img","enc0","img"),("V1_enc0--OUT","enc0","OUT"),
        ("V1_cut--V2_img","cut","img"),
    ]
    for name,a,b in pairs12:
        supp,mult=block_stats(M12,t1,t2,a,b)
        out.append({"image_id":image_id,"r_rectangles":r,"n_nested":n,
                    "interaction":name,"support":supp,"multiplicity":mult})
    return out


def render_grid(A, scale=30):
    h,w=A.shape
    img=Image.new("RGB",(w*scale,h*scale),(255,255,255))
    dr=ImageDraw.Draw(img)
    for r in range(h):
        for c in range(w):
            x0,y0=c*scale,r*scale
            dr.rectangle([x0,y0,x0+scale-1,y0+scale-1], fill=COLOR_RGB[int(A[r,c])])
    for i in range(h+1):
        y=i*scale
        dr.line([(0,y),(w*scale,y)], fill=(120,120,120), width=1)
    for j in range(w+1):
        x=j*scale
        dr.line([(x,0),(x,h*scale)], fill=(120,120,120), width=1)
    return img


def summarize_dataframe(df):
    numeric = [c for c in df.columns if c not in {"image_id"} and pd.api.types.is_numeric_dtype(df[c])]
    rows=[]
    for c in numeric:
        if c in {"seed","r_rectangles","n_nested"}: continue
        x=df[c]
        rows.append({"quantity":c,"mean":x.mean(),"std":x.std(ddof=1),"min":x.min(),"max":x.max()})
    return pd.DataFrame(rows)


def save_bundle(images, primitive_tables, results_df, interactions_df, summary_df, prefix):
    folder=tempfile.mkdtemp(prefix=prefix+"_")
    results_df.to_csv(os.path.join(folder,"per_image_HIG_results.csv"),index=False)
    interactions_df.to_csv(os.path.join(folder,"typed_interactions.csv"),index=False)
    summary_df.to_csv(os.path.join(folder,"summary_statistics.csv"),index=False)
    pd.concat(primitive_tables,ignore_index=True).to_csv(os.path.join(folder,"rectangle_primitives.csv"),index=False)
    np.savez_compressed(os.path.join(folder,"segmentations.npz"), **{f"I_{i:04d}":A for i,A in enumerate(images)})
    # Save first 25 PNGs for visual inspection.
    for i,A in enumerate(images[:25]):
        render_grid(A,scale=24).save(os.path.join(folder,f"image_{i:04d}.png"))
    zip_path=folder+".zip"
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
        for name in os.listdir(folder):
            z.write(os.path.join(folder,name),arcname=name)
    return zip_path


def run_fixed_experiment(r, n, N, seed):
    r=int(r); n=int(n); N=int(N); seed=int(seed)
    if N < 1 or N > 5000:
        raise gr.Error("N must be between 1 and 5000.")
    if not feasible_pair(r,n):
        raise gr.Error(f"The pair (r={r}, n={n}) is not feasible on 10x10 with strict one-pixel nesting margins.")

    images=[]; rows=[]; inter=[]; prim_tables=[]; gallery=[]
    for i in range(N):
        s=seed+i
        A,primitives,depths,layout=generate_rectangular_segmentation(r,n,s)
        row,h=one_row(A,primitives,r,n,s,i)
        images.append(A); rows.append(row); inter.extend(interaction_rows(h,i,r,n))
        ptab=pd.DataFrame(primitives); ptab["image_id"]=i; ptab["seed"]=s
        prim_tables.append(ptab)
        if i<12:
            gallery.append((render_grid(A),f"I{i}: r={r}, n={n}, |V1cut|={row['V1_cut']}, chi_cell={row['chi_cell']}"))

    df=pd.DataFrame(rows)
    idf=pd.DataFrame(inter)
    summary=summarize_dataframe(df)
    inter_summary=(idf.groupby("interaction",as_index=False)[["support","multiplicity"]]
                     .agg(["mean","std","min","max"]))
    inter_summary.columns=["_".join([x for x in col if x]) for col in inter_summary.columns.to_flat_index()]
    inter_summary=inter_summary.rename(columns={"interaction_":"interaction"})

    if not (df["chi_cell"]==2).all():
        raise gr.Error("Euler consistency failure detected.")

    bundle=save_bundle(images,prim_tables,df,idf,summary,f"HIG_r{r}_n{n}_N{N}")
    message=(f"### Completed\n"
             f"- Images: **{N}**\n- Parameters: **r={r}**, **n={n}**\n"
             f"- Euler check: **{int((df.chi_cell==2).sum())}/{N} images with chi_cell=2**\n"
             f"- Mean |V1cut|: **{df.V1_cut.mean():.3f}**\n"
             f"- Mean chi_graph: **{df.chi_graph.mean():.3f}**")
    return message, gallery, df, summary, inter_summary, bundle


def feasible_pairs_up_to(rmax):
    pairs=[]
    for r in range(1,int(rmax)+1):
        for n in range(r):
            if feasible_pair(r,n):
                pairs.append((r,n))
    return pairs


def run_sweep(rmax, reps, seed):
    rmax=int(rmax); reps=int(reps); seed=int(seed)
    pairs=feasible_pairs_up_to(rmax)
    rows=[]
    counter=0
    for r,n in pairs:
        for j in range(reps):
            s=seed+counter
            counter+=1
            A,primitives,depths,layout=generate_rectangular_segmentation(r,n,s)
            row,h=one_row(A,primitives,r,n,s,j)
            rows.append(row)
    df=pd.DataFrame(rows)
    if not (df.chi_cell==2).all():
        raise gr.Error("Euler consistency failure detected in sweep.")

    cols=["V2_img","V1_seg","V1_enc1","V1_enc0","V1_cut","V0_junc","V0_enc",
          "chi_graph","d01","d12","rho_HIG","rho_inc","RAG_E"]
    grouped=df.groupby(["r_rectangles","n_nested"])[cols].agg(["mean","std"]).reset_index()
    grouped.columns=["_".join([str(x) for x in col if str(x)]) for col in grouped.columns.to_flat_index()]

    folder=tempfile.mkdtemp(prefix="HIG_parameter_sweep_")
    df.to_csv(os.path.join(folder,"sweep_per_image.csv"),index=False)
    grouped.to_csv(os.path.join(folder,"sweep_grouped_summary.csv"),index=False)
    zip_path=folder+".zip"
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
        for name in os.listdir(folder): z.write(os.path.join(folder,name),arcname=name)

    msg=(f"### Sweep completed\n- Feasible parameter pairs: **{len(pairs)}**\n"
         f"- Replicates per pair: **{reps}**\n- Total images: **{len(df)}**\n"
         f"- Euler check: **{int((df.chi_cell==2).sum())}/{len(df)}**")
    return msg, grouped, zip_path



In [ ]:

# ============================================================
# Publication experiment: controlled axes
# ============================================================
BASE_SEED = 20260927
N_PER_CONDITION = 100

# A) Rectangle complexity: n=0, r=1,...,8
rect_rows = []
for r in range(1, 9):
    for j in range(N_PER_CONDITION):
        seed = BASE_SEED + r*1000 + j
        A, primitives, depths, layout = generate_rectangular_segmentation(r, 0, seed)
        row, hig = one_row(A, primitives, r, 0, seed, j)
        rect_rows.append(row)
rect_df = pd.DataFrame(rect_rows)

# B) Nesting complexity: r=5, n=0,...,4
nest_rows = []
interaction_data = []
for n in range(5):
    for j in range(N_PER_CONDITION):
        seed = BASE_SEED + n*1000 + j
        A, primitives, depths, layout = generate_rectangular_segmentation(5, n, seed)
        row, hig = one_row(A, primitives, 5, n, seed, j)
        nest_rows.append(row)
        interaction_data.extend(interaction_rows(hig, j, 5, n))
nest_df = pd.DataFrame(nest_rows)
inter_df = pd.DataFrame(interaction_data)

assert (rect_df.chi_cell == 2).all()
assert (nest_df.chi_cell == 2).all()

# Summary tables
rect_metrics = ['V2_img','V1_seg','V1_cut','V0_junc','chi_graph','rho_HIG','rho_inc','RAG_E']
rectangle_table = rect_df.groupby('r_rectangles')[rect_metrics].agg(['mean','std']).reset_index()
rectangle_table.columns = ['_'.join([str(x) for x in c if str(x)]) for c in rectangle_table.columns.to_flat_index()]

nest_metrics = ['V2_img','V1_seg','V1_enc1','V1_enc0','V1_cut','V0_junc','V0_enc','chi_graph','RAG_E','rho_HIG','rho_inc']
nesting_table = nest_df.groupby('n_nested')[nest_metrics].agg(['mean','std']).reset_index()
nesting_table.columns = ['_'.join([str(x) for x in c if str(x)]) for c in nesting_table.columns.to_flat_index()]

coverage_rows = []
for n, g in nest_df.groupby('n_nested'):
    coverage_rows.append({
        'n_nested': n,
        'pct_cut_nonzero': 100*(g.V1_cut > 0).mean(),
        'pct_enc0_nonzero': 100*(g.V1_enc0 > 0).mean(),
        'pct_enc1_nonzero': 100*(g.V1_enc1 > 0).mean(),
        'pct_cut_equals_n': 100*(g.V1_cut == n).mean(),
    })
coverage_table = pd.DataFrame(coverage_rows)

keys = ['V0_junc--V1_seg','V0_enc--V1_enc0','V1_cut--V2_img','V1_seg--V2_img']
interaction_rows_pub = []
for key in keys:
    row = {'interaction': key}
    for n in [0,2,4]:
        x = inter_df[(inter_df.interaction == key) & (inter_df.n_nested == n)]['multiplicity']
        row[f'n{n}_mean'] = x.mean()
        row[f'n{n}_std'] = x.std(ddof=1)
    interaction_rows_pub.append(row)
interaction_table = pd.DataFrame(interaction_rows_pub)

print('Euler check:', int((rect_df.chi_cell == 2).sum() + (nest_df.chi_cell == 2).sum()), '/ 1300')
print('Total image regions:', int(rect_df.V2_img.sum() + nest_df.V2_img.sum()))
print('Total cut generators:', int(rect_df.V1_cut.sum() + nest_df.V1_cut.sum()))

display(rectangle_table)
display(nesting_table)
display(coverage_table)
display(interaction_table)

# Export CSVs
rectangle_table.to_csv('table_rectangle_complexity.csv', index=False)
nesting_table.to_csv('table_nesting_complexity.csv', index=False)
coverage_table.to_csv('table_nesting_coverage.csv', index=False)
interaction_table.to_csv('table_typed_incidence.csv', index=False)
rect_df.to_csv('rectangles_n0_100_each.csv', index=False)
nest_df.to_csv('nesting_r5_100_each.csv', index=False)
inter_df.to_csv('nesting_r5_interactions_100_each.csv', index=False)


In [ ]:

# Write the publication-ready LaTeX section
tex = '\\section{Experimental Evaluation}\n\\label{sec:experimental-evaluation}\n\nThe experimental study is designed to evaluate the intrinsic HIG in a\ncontrolled setting, independently of the HSF realization mechanism and of\napplication-specific prediction tasks.  We use synthetic $10\\times10$\nsegmentations generated from axis-aligned constant-color rectangles.  The\ngenerator has two control parameters: $r$, the total number of rectangular\nprimitives drawn in the image, and $n$, the number of those primitives that\nare drawn strictly inside another rectangle.  Five categorical colors are\navailable, and the drawing order determines the visible segmentation.\n\nThe parameters $(r,n)$ describe the generative process, not the resulting\nHIG.  In particular, overlap and drawing order may change the final number\nof $4$-connected regions, and $n$ is not forced to equal\n$|V_1^{\\mathrm{cut}}|$.  All HIG quantities reported below are therefore\nmeasured outputs of the intrinsic construction.\n\nTwo complementary controlled experiments are considered.  In the first,\nwe vary $r\\in\\{1,\\ldots,8\\}$ with $n=0$ in order to study increasing\nregional complexity without imposed nesting.  In the second, we fix $r=5$\nand vary $n\\in\\{0,1,2,3,4\\}$ in order to isolate the effect of nesting.\nFor each parameter value, $100$ images are generated, using base random\nseed $20260927$.  This gives $800$ images in the rectangle-complexity\nexperiment and $500$ images in the nesting experiment.\n\n\\begin{table}[ht]\n\\centering\n\\caption{Synthetic experimental design.  Each condition contains 100\nindependently generated $10\\times10$ five-label segmentations.}\n\\label{tab:synthetic-design}\n\\begin{tabular}{lcccc}\n\\hline\nExperiment & Fixed parameter & Varied parameter & Conditions & Images \\\\\n\\hline\nRectangle complexity & $n=0$ & $r=1,\\ldots,8$ & 8 & 800 \\\\\nNesting complexity & $r=5$ & $n=0,\\ldots,4$ & 5 & 500 \\\\\n\\hline\nTotal & & & 13 & 1300 \\\\\n\\hline\n\\end{tabular}\n\\end{table}\n\nFor every generated segmentation we compute\n\\[\nS_4(I),\\qquad F,\\qquad HIG(I),\n\\]\nusing the intrinsic construction of Section~\\ref{sec:comb-def-HIG}.  We\nrecord the typed generator counts, the two HIG incidence matrices, their\nsupport and total multiplicity, the RAG size, the graph Euler\ncharacteristic, and the representation-size ratios.  All results below are\nreported as mean $\\pm$ standard deviation over the 100 images in each\ncondition.\n\n%============================================================\n\\subsection{Consistency checks}\n\\label{subsec:synthetic-consistency}\n\nThe implementation performs two exact consistency checks.  First, the\ncut-identification routine matches every frame-free frontier component with\nits regional complementary component through the cut--hole construction.\nSecond, the graded Euler identity\n\\[\n\\chi_{\\mathrm{cell}}(HIG)\n=\n|V_2|-|V_1|+|V_0|\n=\n2\n\\]\nis checked after every HIG construction and is not imposed as an input\nconstraint.\n\nAcross the $1300$ generated images, comprising $7759$ image-region\ngenerators and $1831$ cut generators in total, all cut--hole consistency\nchecks succeeded and\n\\[\n\\chi_{\\mathrm{cell}}(HIG)=2\n\\]\nheld in $1300/1300$ cases.  Thus the synthetic experiment provides an\nimplementation-level verification of the intrinsic construction over a\nlarge collection of distinct segmented configurations.\n\n%============================================================\n\\subsection{Effect of regional complexity}\n\\label{subsec:rectangle-complexity}\n\nTable~\\ref{tab:rectangle-complexity} reports the effect of increasing the\nnumber $r$ of rectangular primitives while keeping $n=0$.  As expected,\nincreasing $r$ increases the number of visible $4$-connected regions and\nthe combinatorial complexity of the frontier organization.\n\n\\begin{table}[ht]\n\\centering\n\\caption{Effect of the number of rectangular primitives for $n=0$.\nValues are mean $\\pm$ standard deviation over 100 images per condition.}\n\\label{tab:rectangle-complexity}\n\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}{c c c c c c c}\n\\hline\n$r$ & $|V_2^{\\mathrm{img}}|$ & $|V_1^{\\mathrm{seg}}|$ &\n$|V_1^{\\mathrm{cut}}|$ & $|V_0^{\\mathrm{junc}}|$ &\n$\\chi_{\\mathrm{graph}}$ & $\\rho_{\\mathrm{HIG}}$ \\\\\n\\hline\n1 & $2.13\\pm0.34$ & $2.64\\pm1.82$ & $0.25\\pm0.44$ & $1.76\\pm1.22$ & $-4.28\\pm2.38$ & $0.020\\pm0.005$ \\\\\n2 & $3.17\\pm0.38$ & $4.89\\pm2.49$ & $0.53\\pm0.64$ & $3.26\\pm1.66$ & $-9.02\\pm3.60$ & $0.032\\pm0.006$ \\\\\n3 & $4.13\\pm0.37$ & $7.52\\pm2.52$ & $0.59\\pm0.71$ & $5.01\\pm1.66$ & $-14.34\\pm3.71$ & $0.044\\pm0.006$ \\\\\n4 & $5.20\\pm0.40$ & $9.83\\pm3.20$ & $0.84\\pm0.86$ & $6.56\\pm2.09$ & $-19.72\\pm4.55$ & $0.057\\pm0.008$ \\\\\n5 & $6.39\\pm0.68$ & $12.45\\pm3.76$ & $0.98\\pm0.96$ & $8.27\\pm2.35$ & $-25.82\\pm5.35$ & $0.070\\pm0.010$ \\\\\n6 & $7.37\\pm0.69$ & $15.35\\pm3.95$ & $1.08\\pm0.97$ & $10.23\\pm2.54$ & $-31.80\\pm5.99$ & $0.084\\pm0.011$ \\\\\n7 & $8.50\\pm0.75$ & $18.27\\pm3.61$ & $1.00\\pm0.86$ & $12.07\\pm2.23$ & $-37.88\\pm5.59$ & $0.097\\pm0.011$ \\\\\n8 & $9.70\\pm0.83$ & $21.31\\pm4.42$ & $1.15\\pm1.01$ & $13.95\\pm2.78$ & $-44.62\\pm6.54$ & $0.112\\pm0.013$ \\\\\n\\hline\n\\end{tabular}}\n\\end{table}\n\nThe mean number of image regions increases from $2.13$ for $r=1$ to\n$9.70$ for $r=8$.  Over the same range,\n$|V_1^{\\mathrm{seg}}|$ increases from $2.64$ to $21.31$ and\n$|V_0^{\\mathrm{junc}}|$ from $1.76$ to $13.95$.  The graph Euler\ncharacteristic becomes increasingly negative, from $-4.28$ to $-44.62$,\nreflecting the increasing cycle complexity of the incidence multigraph.\nThe HIG-to-cubical cell-count ratio remains small in absolute terms but\nincreases from approximately $0.020$ to $0.112$ as the segmentation becomes\nmore complex.\n\nCut generators can occur even when $n=0$, since the random geometry and\noverwriting order may create an enclosing configuration incidentally.\nThus the absence of imposed nesting does not imply the absence of digital\nholes in the final segmentation.\n\n%============================================================\n\\subsection{Effect of controlled nesting}\n\\label{subsec:nesting-complexity}\n\nTo isolate the role of nesting, we fix $r=5$ and increase\n$n$ from $0$ to $4$.  Table~\\ref{tab:nesting-complexity} shows a clear\nreorganization of the HIG: increasing nesting shifts the representation\nfrom junction-rich segmentation frontiers toward enclosing and cut-induced\nstructures.\n\n\\begin{table}[ht]\n\\centering\n\\caption{Effect of controlled nesting for $r=5$.  Values are mean\n$\\pm$ standard deviation over 100 images per condition.}\n\\label{tab:nesting-complexity}\n\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}{c c c c c c c c}\n\\hline\n$n$ & $|V_1^{\\mathrm{seg}}|$ & $|V_1^{\\mathrm{enc},0}|$ &\n$|V_1^{\\mathrm{cut}}|$ & $|V_0^{\\mathrm{junc}}|$ &\n$\\chi_{\\mathrm{graph}}$ & $|E(RAG)|$ & $\\rho_{\\mathrm{HIG}}$ \\\\\n\\hline\n0 & $12.02\\pm4.08$ & $0.93\\pm1.08$ & $1.07\\pm1.10$ & $8.04\\pm2.60$ & $-25.58\\pm5.30$ & $6.85\\pm1.67$ & $0.069\\pm0.010$ \\\\\n1 & $11.27\\pm3.16$ & $1.55\\pm0.81$ & $1.55\\pm0.76$ & $7.49\\pm2.08$ & $-23.78\\pm5.00$ & $6.57\\pm1.75$ & $0.070\\pm0.009$ \\\\\n2 & $8.83\\pm2.00$ & $2.23\\pm0.47$ & $2.23\\pm0.47$ & $5.86\\pm1.28$ & $-20.20\\pm3.27$ & $6.03\\pm1.34$ & $0.065\\pm0.006$ \\\\\n3 & $6.93\\pm1.63$ & $3.04\\pm0.20$ & $3.04\\pm0.20$ & $4.62\\pm1.09$ & $-17.94\\pm3.09$ & $6.28\\pm1.27$ & $0.064\\pm0.007$ \\\\\n4 & $2.28\\pm1.29$ & $4.24\\pm0.43$ & $4.00\\pm0.00$ & $1.52\\pm0.86$ & $-11.04\\pm1.72$ & $4.76\\pm0.43$ & $0.052\\pm0.004$ \\\\\n\\hline\n\\end{tabular}}\n\\end{table}\n\nThe most direct effect is observed in the cut and enclosing layers.\nThe mean number of cut generators increases from $1.07$ at $n=0$ to\nexactly $4.00$ at $n=4$, while the mean number of junction-free enclosing\ngenerators increases from $0.93$ to $4.24$.  Conversely,\n$|V_1^{\\mathrm{seg}}|$ decreases from $12.02$ to $2.28$ and\n$|V_0^{\\mathrm{junc}}|$ from $8.04$ to $1.52$.  Hence the two control\nparameters act on qualitatively different parts of the representation:\n$r$ primarily increases regional and junction complexity, whereas $n$\npromotes enclosing and cut-induced structure.\n\nThe parameter $n$ should not be interpreted as a prescribed HIG invariant.\nFor $n=0$, $63\\%$ of the generated images still contain at least one cut\ngenerator because random rectangle overlap may create a bounded\ncomplementary component.  As $n$ increases, the explicitly nested geometry\nincreasingly dominates this incidental effect.\n\n\\begin{table}[ht]\n\\centering\n\\caption{Occurrence of enclosing and cut structures in the nesting\nexperiment.  Percentages are computed over 100 images per condition.}\n\\label{tab:nesting-coverage}\n\\begin{tabular}{c r r r r}\n\\hline\n$n$ & $V_1^{\\mathrm{cut}}\\neq\\varnothing$ &\n$V_1^{\\mathrm{enc},0}\\neq\\varnothing$ &\n$V_1^{\\mathrm{enc},1}\\neq\\varnothing$ &\n$|V_1^{\\mathrm{cut}}|=n$ \\\\\n\\hline\n0 & 63\\% & 57\\% & 20\\% & 37\\% \\\\\n1 & 100\\% & 100\\% & 4\\% & 57\\% \\\\\n2 & 100\\% & 100\\% & 4\\% & 79\\% \\\\\n3 & 100\\% & 100\\% & 0\\% & 96\\% \\\\\n4 & 100\\% & 100\\% & 0\\% & 100\\% \\\\\n\\hline\n\\end{tabular}\n\\end{table}\n\n%============================================================\n\\subsection{Typed incidence behavior}\n\\label{subsec:typed-incidence-experiment}\n\nThe HIG records not only the presence of generators but also their typed\nincidence multiplicities.  Table~\\ref{tab:typed-incidences} reports selected\nmean multiplicity counts for three nesting levels.  The transition from\n$n=0$ to $n=4$ is visible directly in the incidence structure: junction--\nsegmentation incidences decrease, whereas enclosing and cut incidences\nincrease.\n\n\\begin{table}[ht]\n\\centering\n\\caption{Selected typed HIG incidence multiplicities for $r=5$.\nValues are mean $\\pm$ standard deviation over 100 images.}\n\\label{tab:typed-incidences}\n\\begin{tabular}{lccc}\n\\hline\nInteraction & $n=0$ & $n=2$ & $n=4$ \\\\\n\\hline\n$V_0^{\\mathrm{junc}}\\leftrightarrow V_1^{\\mathrm{seg}}$\n& $24.04\\pm8.16$ & $17.66\\pm3.99$ & $4.56\\pm2.58$ \\\\\n$V_0^{\\mathrm{enc}}\\leftrightarrow V_1^{\\mathrm{enc},0}$\n& $1.86\\pm2.15$ & $4.46\\pm0.94$ & $8.48\\pm0.86$ \\\\\n$V_1^{\\mathrm{cut}}\\leftrightarrow V_2^{\\mathrm{img}}$\n& $2.14\\pm2.21$ & $4.46\\pm0.94$ & $8.00\\pm0.00$ \\\\\n$V_1^{\\mathrm{seg}}\\leftrightarrow V_2^{\\mathrm{img}}$\n& $18.50\\pm6.58$ & $12.71\\pm3.40$ & $3.04\\pm1.72$ \\\\\n\\hline\n\\end{tabular}\n\\end{table}\n\nThe doubled incidence associated with enclosing cycles and cut generators\nis also visible numerically.  For example, at $n=4$ the mean number of cut\ngenerators is $4.00$, while the mean multiplicity of the\n$V_1^{\\mathrm{cut}}\\leftrightarrow V_2^{\\mathrm{img}}$ block is $8.00$.\nLikewise, the mean $V_0^{\\mathrm{enc}}\\leftrightarrow\nV_1^{\\mathrm{enc},0}$ multiplicity is twice the mean number of\njunction-free enclosing generators.\n\n%============================================================\n\\subsection{RAG comparison and representation size}\n\\label{subsec:synthetic-rag-comparison}\n\nThe synthetic benchmark also illustrates that ordinary region adjacency\nand the HIG respond differently to increasing nesting.  For $r=5$, the\nmean number of RAG edges decreases from $6.85$ at $n=0$ to $4.76$ at\n$n=4$, while the mean number of cut generators increases from $1.07$ to\n$4.00$ and the mean number of junction-free enclosing generators from\n$0.93$ to $4.24$.  Thus increasing nesting can simplify the pairwise\nadjacency graph while simultaneously enriching the typed topological layer\nof the HIG.\n\nThis experiment should not be interpreted as a direct empirical proof of\nthe non-injectivity of the HIG-to-RAG forgetful map, because the generated\nimages were not paired so as to enforce RAG isomorphism.  Instead, it shows\nthat the rectangle generator excites structural variables that are absent\nfrom a simple RAG description and provides a controlled test bed for the\nformal strict-refinement result established in Section~\\ref{sec:comb-def-HIG}.\n\nFor all $10\\times10$ images, the underlying cubical model contains\n\\[\n|K_0|+|K_1|+|K_2|=121+220+100=441\n\\]\ncells.  The mean HIG-to-cubical size ratio ranges from approximately\n$0.020$ for the simplest $r=1$ segmentations to $0.112$ for $r=8$.  In the\nnesting experiment it remains between $0.052$ and $0.070$.  These values\nshow that the intrinsic HIG provides a substantial reduction relative to\nthe pixel-level cubical structure for this controlled family, while the\namount of compression decreases as regional complexity increases.\n\n%============================================================\n\\subsection{Scope of the synthetic evaluation}\n\\label{subsec:synthetic-scope}\n\nThe present synthetic evaluation addresses implementation consistency and\nthe response of the intrinsic HIG to controlled geometric structure.  It\ndoes not test two logically separate questions developed elsewhere in the\npaper.  First, the rectangle generator computes the direct HIG and does not\nproduce an independent CA--HSF realization; therefore it cannot establish\nempirical HSF--HIG compatibility.  Second, the synthetic images have no\nexternal semantic class labels, so they cannot support a non-circular\npredictive ROC--AUC comparison between RAG and HIG descriptors.\n\nAccordingly, the results above should be interpreted as evidence that\n(i) the implementation reproduces the intrinsic Euler and cut--hole\nconstraints over a broad controlled family, (ii) the two generator\nparameters affect different typed layers of the HIG in the expected way,\nand (iii) the HIG retains enclosing and cut-induced information whose\nbehavior is not summarized by ordinary pairwise region adjacency.  HSF\ncompatibility and application-level discriminative performance require\nindependent experimental pipelines and should be reported separately if\nthey are included in the final manuscript.\n'
with open("HIG_Experimental_Evaluation_Synthetic.tex", "w", encoding="utf-8") as f:
    f.write(tex)

import zipfile, os
files_to_zip = [
    "HIG_Experimental_Evaluation_Synthetic.tex",
    "table_rectangle_complexity.csv",
    "table_nesting_complexity.csv",
    "table_nesting_coverage.csv",
    "table_typed_incidence.csv",
    "rectangles_n0_100_each.csv",
    "nesting_r5_100_each.csv",
    "nesting_r5_interactions_100_each.csv",
]
with zipfile.ZipFile("HIG_synthetic_experimental_tables_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in files_to_zip:
        z.write(name)
print("Created HIG_synthetic_experimental_tables_bundle.zip")
